In [0]:
my_catalog = dbutils.widgets.get("catalog")
my_schema = dbutils.widgets.get("schema")

In [0]:
%sql
use catalog my_catalog;
use schema bronze_wl;

In [0]:
import pandas

In [0]:
### - Data Cleaning ---

# Strip whitespace from column names
df=spark.sql("select * from customers_sales_silver")
df = df.toDF(*[col.strip() for col in df.columns])

# Standardize customer names to title case and strip whitespace
from pyspark.sql.functions import initcap, trim, col
df = df.withColumn("customer_name", initcap(trim(col("customer_name"))))

# COnver data types for IDs, numeric columns, and dates
from pyspark.sql.functions import to_date

df = df.withColumn("customer_id" , col("customer_id").cast("long"))
df = df.withColumn("units_purchased" , col("units_purchased").cast("double"))
df = df.withColumn("total_price" , col("total_price").cast("double"))
df = df.withColumn("order_date" , to_date(col("order_date")))
                   
# Drop rows with missing values after conversitions
df = df.dropna(subset=["customer_id","units_purchased", "total_price", "order_Date"])

In [0]:
# --- Feature Engineering ---
# Create a new column 'revenue_per_unit' by dividing 'total_price' by 'units_purchased'
from pyspark.sql.functions import year, month, date_format, round as spark_round, col

# Extract order year and order month for time-series analyses
df = df.withColumn("order_year", year(col("order_date")))
df = df.withColumn("order_month", date_format(col("order_date"), "MMM"))

## calculate average price per unit round to 2 decimals
df = df.withColumn("ave_price_per_unit", spark_round(col("total_price")/col("units_purchased"),2))

In [0]:
df.write.mode("overwrite").saveAsTable("customers_Sales_gold")